### Public attributes (no protection at all)

In [ ]:
class Account:
    def __init__(self, owner, balance):
        self.owner = owner
        self.balance = balance

acc = Account("Kavya", 100)
acc.balance = -5000     # nothing stops this
print(acc.balance)      # -5000


### _var (single underscore) — a convention, not a lock

In [ ]:
class Account2:
    def __init__(self, owner, balance):
        self.owner = owner
        self._balance = balance   # single underscore

acc2 = Account2("Kavya", 100)
print(acc2._balance)      # 100 — still readable from outside
acc2._balance = -5000     # still works — Python enforces nothing
print(acc2._balance)      # -5000


What it changes is meaning between programmers: _balance is a signal that says "this is an internal implementation detail — please don't touch this directly from outside, use a proper method instead." It's an honor-system label, not a technical restriction. Python trusts you to respect the underscore; it doesn't check.

### __var — Python rewrites the name for you

In [ ]:
class Account3:
    def __init__(self, owner, balance):
        self.owner = owner
        self.__balance = balance   # written as __balance

acc3 = Account3("Kavya", 100)

print(acc3.__dict__)
# {'owner': 'Kavya', '_Account3__balance': 100}

print(acc3._Account3__balance)   # 100 — this is the REAL name
# print(acc3.__balance)          # AttributeError — this key doesn't exist


acc3.__balance fails — not because it's "protected," but because that literal key genuinely was never created. The real key is _Account3__balance, and that one works completely normally from outside — read it, write it, no restriction. Still not real privacy, just an automatic rename.

### The problem (hypothetical — pretend mangling didn't exist)

In [ ]:
class Base:
    def __init__(self):
        self._id = "base-internal-id"       # single underscore, NOT mangled

class Child(Base):
    def __init__(self):
        super().__init__()
        self._id = "child-internal-id"      # same key, same dict

c = Child()
print(c.__dict__)
# {'_id': 'child-internal-id'}   <- Base's value is just gone


Base is a class you didn't write yourself (a library), and you don't know every internal name it uses.
Base gets updated later (after Child already exists) and happens to add an internal attribute with a name you already picked independently.

In [ ]:
class Base2:
    def __init__(self):
        self.__id = "base-internal-id"      # becomes _Base2__id

class Child2(Base2):
    def __init__(self):
        super().__init__()
        self.__id = "child-internal-id"     # becomes _Child2__id -- DIFFERENT KEY

c2 = Child2()
print(c2.__dict__)
# {'_Base2__id': 'base-internal-id', '_Child2__id': 'child-internal-id'}


### BROKEN — __init__ calls the public method directly

In [ ]:
class MappingBroken:
    def __init__(self, iterable):
        self.items_list = []
        self.update(iterable)          # calls the PUBLIC update

    def update(self, iterable):
        for item in iterable:
            self.items_list.append(item)

class MappingBrokenSubclass(MappingBroken):
    def update(self, keys, values):    # legitimate override, DIFFERENT signature
        for item in zip(keys, values):
            self.items_list.append(item)

m = MappingBrokenSubclass(["a", "b"])   # crashes


Walk through it:

##### MappingBrokenSubclass has no __init__ → the inherited MappingBroken.__init__ runs.
##### It does self.update(iterable). Lookup checks self's actual class first — that's MappingBrokenSubclass — and finds update there immediately. Lookup stops the instant it finds a match; it never even looks at MappingBroken's version.
##### So it calls MappingBrokenSubclass.update(self, keys, values) — needs two arguments — but was only given one (iterable).
##### Crash: TypeError: missing 1 required positional argument: 'values'.

### FIXED — __init__ calls __update, not update

In [ ]:
class Mapping:
    def __init__(self, iterable):
        self.items_list = []
        self.__update(iterable)        # mangled -> self._Mapping__update(iterable)

    def update(self, iterable):
        for item in iterable:
            self.items_list.append(item)

    __update = update    # gives this SAME function a second, private name: _Mapping__update

class MappingSubclass(Mapping):
    def update(self, keys, values):
        for item in zip(keys, values):
            self.items_list.append(item)

m = MappingSubclass(["a", "b"])
print(m.items_list)          # ['a', 'b'] -- __init__ succeeded, no crash

m.update(["k1", "k2"], [1, 2])   # public call -- runs the CHILD's override
print(m.items_list)          # ['a', 'b', ('k1', 1), ('k2', 2)]


#### Why __init__ no longer crashes: self.__update(iterable) was mangled, inside Mapping's class body, to self._Mapping__update(iterable) — a name that only exists because of the __update = update line. MappingSubclass never defined anything called _Mapping__update — it only ever touched the public name update. So lookup searches for _Mapping__update, doesn't find it on MappingSubclass, walks up to Mapping, finds it there — landing safely on Mapping's original one-argument function. No crash.

#### Why the public override still works: m.update(["k1","k2"], [1,2]) is called from outside using the public name. That lookup still checks self's actual class first, finds MappingSubclass.update, and correctly runs the child's two-argument version. Overriding still works exactly as intended for outside callers.

### super() vs self.__method — opposite directions

In [ ]:
class Parent:
    def show(self):
        print("Parent.show")

class Child3(Parent):
    def show(self):
        super().show()          # child deliberately calling parent's version
        print("Child3.show")

Child3().show()
# Parent.show
# Child3.show


Child wants to deliberately call the parent's version → super().method_name(...). This is the everyday, normal tool. Nothing to do with mangling at all.


Parent wants to guarantee it always runs its OWN version, immune to being overridden by any child → self.__method_name(...) inside the parent's own class body (mangled). This is the opposite direction — the base class protecting its own internal call, not a subclass reaching upward.

| Who's calling | What they want | How |
|---|---|---|
| Child calling parent's version | Run the parent's code, on purpose | `super().method(...)` |
| Parent calling its own version | Always run MY code, no matter what a subclass did to the public name | `self.__method(...)` (mangled) |

### The one lookup rule

In [ ]:
class Demo:
    def a(self):
        self.b()          # calls whatever "b" resolves to on self's actual class

    def b(self):
        print("Demo.b")

class DemoChild(Demo):
    def b(self, x):        # overridden with a DIFFERENT signature — Python allows this, no check
        print("DemoChild.b", x)

d = DemoChild()
d.a()    # crashes: TypeError, b() missing required argument 'x'


self.NAME(...) always works the same way, no exceptions:

1. Check self's actual class first, then walk up the MRO one class at a time.
2. Stop at the very first class that defines NAME. No further checking — the search doesn't care if a different class higher up also has it.
3. Only after a match is found does the call happen — and only then, at call time, does an argument mismatch (if any) surface as a separate, later TypeError.

Two things that follow directly from this:
- There is no "try the next one" fallback, ever, for any method, mangled or not. Once lookup finds a match, that's final.
- Python never enforces matching signatures on an override. You're free to override b(self) with b(self, x) — no error at definition time. It only breaks later, when something calls it assuming the old signature.

That absence of a safety net is exactly why mangling exists: since Python won't stop an incompatible override and won't fall back to the parent automatically, the parent has to proactively protect its own internal call by giving it a name (mangled) the search will never confuse with the public one.

### 9a: You only need a private helper — no public version at all

In [9]:
class Widget:
    def __init__(self, data):
        self.__setup(data)          # only ever called internally

    def __setup(self, data):        # never meant to be public to begin with
        self.data = list(data)

w = Widget([1, 2, 3])
print(w.data)      # [1, 2, 3]


[1, 2, 3]


Widget.__setup never had that dual role. It was never public to begin with — nobody calls it except Widget itself, from inside its own methods. So there's nothing to "protect it from" in the collision sense — no public version exists that a subclass could confuse it with, and no public version a subclass is expected to override. You just write it as __setup from the start because that's simply what it always was: a private implementation detail, never a two-name situation.

If a method was never going to be public in the first place — just name it __ directly, right from the definition. No aliasing needed here, because there's no separate public version to protect it from.

### You need BOTH a public, override-able version AND a protected copy of the original

In [ ]:
class Mapping:
    def update(self, iterable):     # public — subclasses CAN freely override this
        ...

    __update = update               # private alias to the ORIGINAL, un-overridden version
